In [ ]:
# Проверка порта
import socket
sock = socket.create_connection(("data-flow-postgres-airflow-1", 5432), timeout=3)
print("✅ Порт 5432 доступен")
sock.close()

In [ ]:
from pyspark.sql import SparkSession

# 1. Создаём сессию с JDBC-драйвером
spark = (SparkSession.builder
    .appName("DE_Task_PG_Connection")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.executor.instances", "2")
    .getOrCreate())

# 2. Параметры подключения
jdbc_url = "jdbc:postgresql://postgres-source:5432/source"

props = {
    "user": "source",
    "password": "source",  # ← замени на реальный из docker-compose
    "driver": "org.postgresql.Driver"
}

# 3. Чтение таблицы
df = spark.read.jdbc(url=jdbc_url, table="d_raw.raw_events", properties=props)

# 4. Вывод таблицы
print(f"✅ Загружено строк: {df.count()}")
df.show(5, truncate=False)

In [ ]:
df.createOrReplaceTempView("raw_events")

df1 = spark.sql("""
SELECT *
FROM raw_events
""")

df1.show()

In [ ]:
df1.show()

In [ ]:
spark.stop()